# 동해안 참다랑어 어장 변화와 TAC 제도 비판 — 웹 수집 리포트

**분석 질문**: 동해 표층수온 상승 추세와 참다랑어(다랑어류) 어획량 증가 추세는 함께 나타나는가, 그리고 그 증가가 현행 TAC 소진 시점과 시기적으로 충돌하는가?

**배경**: 기후변화로 인한 해수온 상승으로 참다랑어의 회유 경로와 분포 밀도가 근본적으로 변화했음에도, 현행 TAC 제도는 과거 통계·정적 할당 방식에 묶여 있어 자원 보호와 어업인 생계 지원 모두에서 한계를 보인다는 문제의식에서 출발한다.

**필요 컬럼**: 연-월, 동해 표층수온(평균/평년/편차), 참다랑어(다랑어류) 어획량·생산금액, (보조) 관련 논문 연도·인용수·주제

**수집 대상 및 방법**
| 데이터 | 대상 URL / 출처 | 방법 |
|---|---|---|
| 동해 표층수온 | https://oceanclimate.kr/sst/?start=2005-01&end=2026-05&area=all&var=mean&month=all | requests + BeautifulSoup |
| 참다랑어(다랑어류) 어획량 | KOSIS 어업생산동향조사 (어업별 품종별 통계) | 공식 CSV 다운로드 (조회 셀 2만 개 초과 제한으로 스크래핑 대신 사용) |
| 관련 논문 트렌드 (보조) | OpenAlex API | requests (공식 오픈 API) |

**윤리 점검**
- robots.txt 확인 완료 (oceanclimate.kr)
- 요청 간 `time.sleep()` 적용
- 로그인/개인정보 페이지 미사용
- Google Scholar는 robots.txt 및 이용약관상 자동화 접근이 금지되어 있어 수집 대상에서 제외, 대신 공식 오픈 API인 OpenAlex 사용

## 1. 라이브러리 불러오기

In [ ]:
import requests
import time
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup

plt.rcParams['font.family'] = 'AppleGothic'  # Windows는 'Malgun Gothic'으로 변경
plt.rcParams['axes.unicode_minus'] = False

sns.set_style('whitegrid')

## 2. 수집 A — 동해 표층수온 (BeautifulSoup)

TODO: 아래 셀렉터(`table`, `tbody tr`, `td`)는 실제 페이지 소스를 확인한 뒤 정확히 맞춰야 합니다.
실행 전에 `resp.status_code`가 200인지, `len(rows)`가 0이 아닌지 반드시 확인하세요 (조용한 실패 방지).

In [ ]:
sst_url = "https://oceanclimate.kr/sst/?start=2005-01&end=2026-05&area=all&var=mean&month=all"
headers = {"User-Agent": "Mozilla/5.0 (educational scraping practice)"}

resp = requests.get(sst_url, headers=headers, timeout=10)
print("status:", resp.status_code)

soup = BeautifulSoup(resp.text, "html.parser")

# TODO: 실제 페이지 소스를 보고 정확한 선택자로 교체하세요.
table = soup.select_one("table")
rows = table.select("tbody tr") if table else []
print("추출된 행 개수:", len(rows))  # 검증

In [ ]:
# TODO: 실제 컬럼 순서에 맞게 인덱스를 조정하세요.
sst_records = []
for row in rows:
    cols = [td.get_text(strip=True) for td in row.select("td")]
    if not cols:
        continue
    sst_records.append({
        "date_raw": cols[0] if len(cols) > 0 else None,
        "donghae_mean": cols[9] if len(cols) > 9 else None,   # 동해평균 위치 예상 (스크린샷 기준, 확인 필요)
        "donghae_normal": cols[10] if len(cols) > 10 else None,  # 동해평년
        "donghae_anomaly": cols[11] if len(cols) > 11 else None,  # 동해편차
    })

sst_raw_df = pd.DataFrame(sst_records)
print(len(sst_raw_df))
sst_raw_df.to_csv("sst_raw.csv", index=False)
sst_raw_df.head()

## 3. 수집 B — 참다랑어(다랑어류) 어획량 (KOSIS CSV)

KOSIS는 조회 결과가 2만 셀을 초과하여 화면 조회/오픈API 대신 공식 CSV 다운로드 기능을 사용했습니다.
다운로드한 파일을 이 노트북과 같은 폴더에 두고 아래에서 불러옵니다.

In [ ]:
# TODO: 실제 다운로드한 파일명으로 교체하세요.
kosis_filename = "어업별_품종별_통계.csv"

for enc in ["cp949", "euc-kr", "utf-8-sig"]:
    try:
        catch_raw_df = pd.read_csv(kosis_filename, encoding=enc)
        print(f"성공: encoding={enc}")
        break
    except Exception as e:
        print(f"실패 ({enc}):", e)

print(catch_raw_df.columns.tolist())
catch_raw_df.head()

## 4. 수집 C (보조) — 수산자원·기후변화 관련 논문 트렌드 (OpenAlex API)

In [ ]:
openalex_url = "https://api.openalex.org/works"
queries = [
    "bluefin tuna distribution climate change",
    "fish stock shift warming ocean",
    "adaptive TAC quota fisheries climate",
]

all_papers = []
for q in queries:
    resp = requests.get(
        openalex_url,
        params={"search": q, "per_page": 50, "sort": "publication_year:desc"},
        headers={"User-Agent": "mailto:your_email@example.com"},
    )
    print(q, resp.status_code)  # 검증
    if resp.status_code == 200:
        data = resp.json()
        for w in data.get("results", []):
            all_papers.append({
                "title": w.get("title"),
                "year": w.get("publication_year"),
                "citations": w.get("cited_by_count"),
                "concepts": [c["display_name"] for c in w.get("concepts", [])[:5]],
            })
    time.sleep(2)  # 요청 예의

papers_df = pd.DataFrame(all_papers).drop_duplicates(subset="title")
print("수집된 논문 수:", len(papers_df))
papers_df.to_csv("papers_raw.csv", index=False)
papers_df.head()

## 5. 데이터 정제

### 5-1. 수온 데이터 정제

In [ ]:
sst_df = sst_raw_df.copy()

# TODO: 실제 날짜 형식에 맞게 파싱 (예: '1981년 01월' -> 1981-01)
# sst_df['year_month'] = pd.to_datetime(...)

for col in ["donghae_mean", "donghae_normal", "donghae_anomaly"]:
    sst_df[col] = pd.to_numeric(sst_df[col], errors="coerce")

print("정제 전:", len(sst_raw_df), "행 / 정제 후 결측 개수:", sst_df["donghae_mean"].isna().sum())
sst_df.head()

### 5-2. 어획량 데이터 정제

In [ ]:
catch_df = catch_raw_df.copy()

# TODO: 실제 컬럼명에 맞게 수정
# - 콤마 제거 후 숫자 변환
# - 결측('-', 'X' 등) 처리 및 근거 메모

catch_df.head()

### 5-3. 논문 데이터 정제

In [ ]:
papers_clean_df = papers_df.dropna(subset=["year"]).copy()
papers_clean_df["year"] = papers_clean_df["year"].astype(int)
papers_clean_df = papers_clean_df[papers_clean_df["year"] >= 2000]

print("정제 전:", len(papers_df), "-> 정제 후:", len(papers_clean_df))
papers_clean_df.head()

## 6. 시각화

### 6-1. 연도별 동해 수온 편차 추세

In [ ]:
# TODO: year_month/year 컬럼 완성 후 실행
# fig, ax = plt.subplots(figsize=(10, 5))
# sns.lineplot(data=sst_df, x='year', y='donghae_anomaly', ax=ax)
# ax.set_title('연도별 동해 표층수온 편차 추세')
# ax.set_xlabel('연도'); ax.set_ylabel('수온 편차 (°C)')
# plt.show()

# 해석: (그래프 확인 후 1~2문장 작성)

### 6-2. 연도별 참다랑어(다랑어류) 어획량 추세

In [ ]:
# TODO
# 해석: (작성)

### 6-3. 월별 어획량 분포 (TAC 소진 시점 추정)

In [ ]:
# TODO: boxplot으로 월별 어획량 분포 확인
# 해석: (작성)

### 6-4. (보조) 연도별 관련 논문 발표 수 추이 및 주요 키워드

In [ ]:
year_counts = papers_clean_df["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
year_counts.plot(kind="bar", ax=ax)
ax.set_title("연도별 기후변화-어류분포 관련 논문 수")
ax.set_xlabel("연도")
ax.set_ylabel("논문 수")
plt.tight_layout()
plt.show()

# 해석: (작성)

In [ ]:
all_concepts = []
for c_list in papers_clean_df["concepts"]:
    all_concepts.extend(c_list)

top_concepts = Counter(all_concepts).most_common(10)
concepts_df = pd.DataFrame(top_concepts, columns=["concept", "count"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=concepts_df, x="count", y="concept", ax=ax)
ax.set_title("관련 논문 상위 키워드(주제)")
plt.tight_layout()
plt.show()

# 해석: (작성)

## 7. 마무리 정리

**처음 질문** → 동해 표층수온 상승과 참다랑어 어획량 증가는 함께 나타나는가, TAC 소진 시점과 충돌하는가?

**그린 그래프** → (1) 연도별 수온 편차 추세 (2) 연도별 어획량 추세 (3) 월별 어획량 분포 (4) 관련 논문 트렌드

**알게 된 점** → (그래프 결과를 바탕으로 작성)

**이번 분석이 보여주지 못한 것** → 
- 수온-어획량 간 상관관계는 확인했으나 인과관계까지는 증명하지 못함
- TAC 소진 정확한 시점(고시 데이터)은 별도로 확보하지 못해, 월별 어획량 급감 패턴으로 간접 추정함
- 논문 수 증가는 학계 관심의 정성적 근거일 뿐, 통계적 인과 증거는 아님
- 향후 GAM/MaxEnt 기반 동적 분포모델로 확장 시 수온·염분·먹이생물 데이터가 추가로 필요

## 8. 윤리 점검 리포트

1. **수집 개요**: 동해 표층수온(oceanclimate.kr, BeautifulSoup), 참다랑어 어획량(KOSIS, 공식 CSV 다운로드), 관련 논문(OpenAlex, 공식 API)
2. **수집 결과**: (수치는 위 셀 출력 결과로 채우기)
3. **데이터 신뢰성**: 추출 행 수를 개수 검증(`len()`)으로 확인함. 결측치는 근거를 남기고 처리함.
4. **윤리 점검**:
   - oceanclimate.kr robots.txt 확인 완료, 허용된 경로만 수집
   - 요청 간 `time.sleep()` 적용
   - KOSIS는 조회 셀 제한으로 공식 CSV 다운로드 사용 (스크래핑 아님, 정당한 사유 명시)
   - Google Scholar는 robots.txt/이용약관상 자동화 접근 금지 대상이라 수집하지 않았고, 대신 공식 오픈 API인 OpenAlex 사용
   - 로그인 필요 페이지·개인정보 미수집